# Setup

In [ ]:
# load required libraries with requirements.txt
! pip install -r requirements.txt

In [3]:
#Initial imports

import matplotlib.pyplot as plt
import mysql.connector as mysql
import pandas as pd
import numpy as np
import json

from DbConnector import DbConnector

In [ ]:
# Import datasets
credits_df = pd.read_csv("movies_ds/credits.csv")
keywords_df = pd.read_csv("movies_ds/keywords.csv")
links_small_df = pd.read_csv("movies_ds/links_small.csv")
links_df = pd.read_csv("movies_ds/links.csv")
movies_metadata_df = pd.read_csv("movies_ds/movies_metadata.csv")
ratings_small_df = pd.read_csv("movies_ds/ratings_small.csv")
ratings_df = pd.read_csv("movies_ds/ratings.csv")

In [ ]:
# Cleaning of the data
credits_df_clean = credits_df.copy()
keywords_df_clean = keywords_df.copy()
links_small_df_clean = links_small_df.copy()
links_df_clean = links_df.copy()
movies_metadata_df_clean = movies_metadata_df.copy()
ratings_small_df_clean = ratings_small_df.copy()
ratings_df_clean = ratings_df.copy()

# Getting rid of duplicates (movies with the same id)
movies_metadata_df_clean = movies_metadata_df_clean.drop_duplicates(subset='id')

# Getting rid of movies with unknown release date
movies_metadata_df_clean = movies_metadata_df_clean.dropna(axis=0, subset='release_date')

In [ ]:
from datetime import datetime

def timestamp_to_datetime(timestamp):
    """
    In: timestamp (number of seconds since January 1st, 1970)
        typically items from the 'timestamp' column in the ratings_df dataset are in this format
    Out: datetime following this format: YYYY-MM-DD HH:MM:SS
    """
    return datetime.fromtimestamp(timestamp)

def date_to_datetime(date):
    """
    In: date following this format: YYYY-MM-DD as a string
        typically items from the 'release_date' column in the movies_metadata_df dataset are in this format
    Out: datetime following this format: YYYY-MM-DD HH:MM:SS
    """
    return datetime.strptime(date, "%Y-%m-%d")

# Getting rid of reviews posted before the movie release date
# Only doable for ratings_small_df, takes too long to compute for ratings_df
nb_ratings_deleted = 0
print(len(ratings_small_df_clean))
i = 0
for index, row in ratings_small_df_clean.iterrows():
    rating_datetime = timestamp_to_datetime(row['timestamp'])
    movie_id = str(int(row['movieId']))
    movie = movies_metadata_df_clean.loc[movies_metadata_df_clean['id'] == movie_id]
    if not movie.empty:
        movie = movie.iloc[0]
        movie_release_datetime = date_to_datetime(str(movie['release_date']))
        if rating_datetime < movie_release_datetime:
            ratings_small_df_clean.drop(index)
            nb_ratings_deleted += 1
    i += 1
    print(i)
print(str(nb_ratings_deleted) + " ratings deleted")

In [ ]:
from DbConnector import DbConnector
import pymongo

class DbConnector:
    """
    Connects to the MongoDB server on the Ubuntu virtual machine.
    Connector needs HOST, USER and PASSWORD to connect.

    Example:
    HOST = "tdt4225-00.idi.ntnu.no" // Your server IP address/domain name
    USER = "testuser" // This is the user you created and added privileges for
    PASSWORD = "test123" // The password you set for said user
    """

    def __init__(self,
                 DATABASE='DATABASE_NAME',
                 HOST="tdt4225-03.idi.ntnu.no",
                 USER="TEST_USER",
                 PASSWORD="test123"):
        uri = "mongodb://%s:%s@%s:%d/%s?authSource=%s" % (USER, PASSWORD, HOST, PORT, DATABASE, DATABASE)
        # Connect to the databases
        try:
            self.client = MongoClient(uri)
            self.db = self.client[DATABASE]
        except Exception as e:
            print("ERROR: Failed to connect to db:", e)

        # get database information
        print("You are connected to the database:", self.db.name)
        print("-----------------------------------------------\n")

    def close_connection(self):
        # close the cursor
        # close the DB connection
        self.client.close()
        print("\n-----------------------------------------------")
        print("Connection to %s-db is closed" % self.db.name)

class Program:

    def __init__(self):
        self.connection = DbConnector()
        self.client = self.connection.client
        self.db = self.connection.db

    def create_coll(self, collection_name):
        collection = self.db.create_collection(collection_name)    
        print('Created collection: ', collection)

    def insert_documents(self, collection_name):
        docs = [
            {
                "_id": 1,
                "name": "Bobby",
                "courses": 
                    [
                    {'code':'TDT4225', 'name': ' Very Large, Distributed Data Volumes'},
                    {'code':'BOI1001', 'name': ' How to become a boi or boierinnaa'}
                    ] 
            },
            {
                "_id": 2,
                "name": "Bobby",
                "courses": 
                    [
                    {'code':'TDT02', 'name': ' Advanced, Distributed Systems'},
                    ] 
            },
            {
                "_id": 3,
                "name": "Bobby",
            }
        ]  
        collection = self.db[collection_name]
        collection.insert_many(docs)
        
    def fetch_documents(self, collection_name):
        collection = self.db[collection_name]
        documents = collection.find({})
        for doc in documents: 
            pprint(doc)
        

    def drop_coll(self, collection_name):
        collection = self.db[collection_name]
        collection.drop()

        
    def show_coll(self):
        collections = self.client['test'].list_collection_names()
        print(collections)

    def check_data(self):
        movies = self.db["Movies"]
        movies_sample = movies.find({}).limit(10)
        for movie in movies_sample: 
            pprint(movie)

        contributors = self.db["Contributors"]
        contributors_sample = contributors.find({}).limit(10)
        for contributor in contributors_sample: 
            pprint(contributor)

        raters = self.db["Raters"]
        raters_sample = raters.find({}).limit(10)
        for rater in raters_sample: 
            pprint(rater)

    def exercise_5(self):
        collection = self.db["Movies"]

        pipeline = [
            {},     # Getting all the movies
            {       # Adding useful fields (decade / primary genre / runtime as double)
                "$addFields": {
                    "decade": {
                        {"$multiply": [                                                 # Getting the decade from the first 2 digits
                            {"$floor": {"$divide": [                                    # Getting the first 2 digits of the decade
                                {"$toInt": {"$substrCP": ["$release_date", 0, 4]}},     # Getting the year of release
                                100]}}, 
                            100]}
                    },
                    "$primary_genre": {
                        "$ifNull": [
                            {"$getField": {
                                "field": "name",
                                "input": {"$arrayElemAt": ["$genres", 0]}               # Getting primary genre if it exists
                            }},
                            "Unknown"
                        ]
                    },
                    "$runtime_double": {"$toDouble": "$runtime"}
                }
            },
            {       # Grouping movies by (decade, primary_genre)
                "$group": {
                    "_id": {
                        "decade": "$decade",
                        "primary_genre": "primary_genre"
                    },
                    "movie_count": {"$sum": 1},
                    "runtimes": {"$push": "$runtime_double"}
                }
            },
            {       # Getting all the runtimes for each group of movies
                "$project": {
                    "_id": 0,
                    "decade": "_id.decade",
                    "primary_genre": "$_id.primary_genre",
                    "movie_count": 1,
                    "runtimes": {
                        "$sortArray": {
                            "input": "runtimes",
                            "sortBy": 1
                        }
                    }
                }
            },
            {       # Calculating the median runtime for each group of movies
                "$project": {
                    "decade": 1,
                    "primary_genre": 1,
                    "movie_count": 1,
                    "median_runtime": {
                        "$median": {
                            "input": "$runtimes",
                            "method": "approximate"
                        }
                    }
                }
            },
            {       # Proper decade label (1980s instead of 1980)
                "$addFields": {
                    "decade_label": {
                        "$concat": [{"$toString": "$decade"}, "s"]
                    }
                }
            },
            {       # Sorting the result
                "$sort": {
                    "decade": 1,        # ascending sort
                    "runtime": -1,      # descending sort
                    "primary_genre": 1  # ascending sort
                }
            }
        ]

        result = list(collection.aggregate(pipeline))

        for doc in results:
            print(doc)

    def exercise_6(self):
        collection = self.db["Movies"]

        pipeline = [
            {}, # Getting all the movies
            {   # Getting all the (movie, cast member id) pairs
                "$unwind": "$cast"
            },
            {   # Joining the (movie, cast member id) pairs to the associated cast member
                "$lookup": {
                    "from": "Contributors",
                    "localField": "cast.contributor_id",
                    "foreignField": "_id",
                    "as": "cast"
                }
            },
            {
                "$unwind": "$cast"
            },
            {   # Adding useful fields (decade / info on contributor's gender (is female / is known))
                "$addFields": {
                    "decade": {
                        {"$multiply": [                                                 # Getting the decade from the first 2 digits
                            {"$floor": {"$divide": [                                    # Getting the first 2 digits of the decade
                                {"$toInt": {"$substrCP": ["$release_date", 0, 4]}},     # Getting the year of release
                                100]}}, 
                            100]}
                    },
                    "is_female": {
                        "$eq": ["$contributor.gender", 1]
                    },
                    "has_known_gender": {
                        "$ne": ["$contributor.gender", 0]
                    }
                }
            },
            {   # Getting number of women and contributors whose gender is known for each movie
                "$group": {
                    "_id": "$_id",
                    "decade": {"$first": "$decade"},    # all (movie, cast) pairs have the same decade when the movie is the same so only fetching the decade once in a movie group
                    "female_count": {
                        "$sum": {"$cond": ["$is_female", 1, 0]}     # if contributor is_female add 1, else add 0
                    },
                    "has_known_gender_count": {
                        "$sum": {"$cond": ["$has_known_gender", 1, 0]}
                    }
                }
            },
            {   # Avoiding division by zero in next steps of the pipeline
                "$match": {
                    "known_gender_count": {"$gt": 0}
                }
            },
            {   # Adding field for the proportion of female cast members for each movie
                "$addFields": {
                    "female_proportion": {
                        "$divide": ["$female_count", "$known_gender_count"]
                    }
                }
            },
            {   # Grouping by decade
                "$group": {
                    "_id": "$decade",
                    "movie_count": {"$sum": 1},
                    "average_female_proportion": {"$avg": "$female_proportion"}
                }
            },
            {   # Sorting by average female proportion
                "$sort": {
                    "average_female_proportion": -1,    # Descending sort
                    "_id": 1
                }
            },
            {   # Expected result
                "$project": {
                    "_id": 0,
                    "decade": "$_id",
                    "movie_count": 1,
                    "average_female_proportion": {"$round": ["$average_female_proportion", 4]}
                }
            }
        ]

        result = list(collection.aggregate(pipeline))

        for doc in results:
            print(doc)

    def exercise_7(self):
        collection = self.db["Movies"]

        pipeline = [
            {   # Filtering movies with a vote count greater than 50 and overview or tagline matching "noir" or "neo-noir"
                "$match": {"$and": [
                        {"vote_count": {"$gt": 50}},
                        {"$or": [
                            {"$overview": {"$regex": "/*noir*/"}},    # only searching for the pattern "noir" because if it contains "neo-noir" then it contains "noir"
                            {"$tagline": {"$regex": "/*noir*/"}}
                        ]}
                    ]
                }
            },
            {   # Adding useful field (year of release)
                "$addingFields": {
                    "year": {"$substrCP": ["$release_date", 0, 4]}
                }
            },
            {   # Sorting by vote average
                "$sort": {
                    "$vote_average": -1     # Descending sort
                }
            },
            {   # Only returning the 20 highest movies on the list
                "$limit": 20
            },
            {   # Expected result
                "$project": {
                    "_id": 0,
                    "title": 1,
                    "year": 1,
                    "vote_average": 1,
                    "vote_count": 1
                }
            }
        ]

        result = list(collection.aggregate(pipeline))

        for doc in results:
            print(doc)


def main():
    program = None
    try:
        program = Program()

        # program.initiate_db(df_clean)  WILL RESET THE DATABASE

        program.check_data()
        # program.show_tables()

        # program.exercise_1()
        
    except Exception as e:
        print("ERROR: Failed to use database:", e)
    finally:
        if program:
            program.connection.close_connection()

if __name__ == '__main__':
    main()
    print("Done")